# Extrator de avaliações PT-BR: demo

Qwen2.5-0.5B ajustado com QLoRA para transformar avaliações de clientes em JSON (sentimento, aspectos, problema principal, necessidade de ação).

- Modelo: https://huggingface.co/aitalo/qwen2.5-0.5b-avaliacoes-ptbr
- Código: https://github.com/ItaloSLeao/pt-review-extractor

Para rodar: **Ambiente de execução → Alterar tipo de ambiente → GPU T4** (funciona também em CPU, mais devagar) e depois **Executar tudo**.

In [ ]:
!pip -q install transformers gradio

## 1. Prompt e validação (mesmo código de `src/common.py` do repositório)

In [ ]:
"""Schema, prompt e parsing compartilhados por todas as etapas."""
import json
import re

ASPECTS = ["entrega", "produto", "atendimento", "preco"]
SENTIMENTS = ["positivo", "neutro", "negativo"]
ASPECT_VALUES = ["positivo", "negativo", None]

SYSTEM_PROMPT = """Você extrai informações estruturadas de avaliações de clientes em português.
Responda APENAS com um objeto JSON válido, sem texto adicional, no formato:
{
  "sentimento": "positivo" | "neutro" | "negativo",
  "aspectos": {
    "entrega": "positivo" | "negativo" | null,
    "produto": "positivo" | "negativo" | null,
    "atendimento": "positivo" | "negativo" | null,
    "preco": "positivo" | "negativo" | null
  },
  "problema_principal": string curta (até 10 palavras) | null,
  "requer_acao": true | false
}
Regras:
- Use null para aspectos não mencionados.
- "entrega" cobre prazo, atraso, rastreio e não recebimento.
- "problema_principal" é null quando não há reclamação.
- "requer_acao" é true quando a empresa precisa agir (produto não recebido, defeito, item errado, pedido de contato)."""


def build_messages(text: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Avaliação: {text.strip()}"},
    ]


def parse_output(raw: str) -> dict | None:
    """Extrai e valida o JSON. Retorna None se inválido."""
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    return obj if is_valid(obj) else None


def is_valid(obj) -> bool:
    if not isinstance(obj, dict):
        return False
    if obj.get("sentimento") not in SENTIMENTS:
        return False
    asp = obj.get("aspectos")
    if not isinstance(asp, dict) or set(asp) != set(ASPECTS):
        return False
    if any(asp[a] not in ASPECT_VALUES for a in ASPECTS):
        return False
    if not isinstance(obj.get("requer_acao"), bool):
        return False
    pp = obj.get("problema_principal")
    return pp is None or isinstance(pp, str)


def to_json(obj: dict) -> str:
    return json.dumps(obj, ensure_ascii=False)


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

## 2. Carregar o modelo

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "aitalo/qwen2.5-0.5b-avaliacoes-ptbr"
tok = AutoTokenizer.from_pretrained(MODEL)
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype, device_map="auto").eval()
print("dispositivo:", model.device)


def extrair(texto):
    texto = (texto or "").strip()
    if not texto:
        return {"erro": "digite uma avaliação"}
    prompt = tok.apply_chat_template(build_messages(texto), tokenize=False, add_generation_prompt=True)
    enc = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=200, do_sample=False)
    raw = tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True)
    obj = parse_output(raw)
    return obj if obj is not None else {"erro": "saída inválida", "raw": raw}

## 3. Exemplos

In [ ]:
exemplos = [
    "Chegou antes do prazo e o produto é excelente!",
    "Comprei 2 unidades e só veio 1. Ninguém responde no SAC.",
    "O produto é bom, mas o frete foi absurdo de caro.",
    "Quero trocar, veio na cor errada.",
]
for t in exemplos:
    print(t, "\n ->", extrair(t), "\n")

## 4. Interface interativa (gera um link público temporário)

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=extrair,
    inputs=gr.Textbox(lines=4, label="Avaliação do cliente"),
    outputs=gr.JSON(label="Extração estruturada"),
    examples=exemplos,
    title="Extrator de avaliações PT-BR",
)
demo.launch(share=True)